# Comparación entre Ciclos — 2025-2 vs 2026-1

Notebook para comparar el rendimiento de los alumnos entre ciclos académicos.
Permite identificar tendencias, evaluar la efectividad de cambios pedagógicos
y analizar alumnos que repiten el curso.

**Prerequisito:** Haber ejecutado `2026-1_exploracion.ipynb` y tener
los archivos de salida en `data/*/output/`.

## 1. Imports y configuración

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar directorio raíz al path
RAIZ = Path().resolve().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.reporte import comparar_ciclos, grafica_comparacion_ciclos, resumen_curso

# Paths de los archivos de salida de cada ciclo
ARCHIVO_2025_2 = RAIZ / "data" / "2025-2" / "output" / "notas_calculadas_2025-2.csv"
ARCHIVO_2026_1 = RAIZ / "data" / "2026-1" / "output" / "notas_calculadas_2026-1.csv"

print(f"Archivo 2025-2: {'✅' if ARCHIVO_2025_2.exists() else '❌ No encontrado'} — {ARCHIVO_2025_2}")
print(f"Archivo 2026-1: {'✅' if ARCHIVO_2026_1.exists() else '❌ No encontrado'} — {ARCHIVO_2026_1}")

## 2. Cargar outputs de cada ciclo

In [ ]:
# Cargar notas del ciclo 2025-2
if ARCHIVO_2025_2.exists():
    df_2025_2 = pd.read_csv(ARCHIVO_2025_2)
    print(f"2025-2: {len(df_2025_2)} alumnos, columnas: {list(df_2025_2.columns)}")
else:
    print("⚠️ Archivo 2025-2 no encontrado. Creando DataFrame vacío de ejemplo.")
    df_2025_2 = pd.DataFrame(columns=["Código", "Correo", "EA1", "EA2", "EA3", "NF"])

# Cargar notas del ciclo 2026-1
if ARCHIVO_2026_1.exists():
    df_2026_1 = pd.read_csv(ARCHIVO_2026_1)
    print(f"2026-1: {len(df_2026_1)} alumnos, columnas: {list(df_2026_1.columns)}")
else:
    print("⚠️ Archivo 2026-1 no encontrado. Ejecuta primero 2026-1_exploracion.ipynb")
    df_2026_1 = pd.DataFrame(columns=["Código", "Correo", "EA1", "EA2", "EA3", "NF"])

## 3. Comparar medias por EA (EA1...EA3)

In [ ]:
# Evaluaciones a comparar entre ciclos
# Ajustar según qué columnas existan en ambos ciclos
EVALUACIONES_COMUNES = ["EA1", "EA2", "EA3"]
EVALUACIONES_COMUNES = [
    e for e in EVALUACIONES_COMUNES
    if e in df_2025_2.columns and e in df_2026_1.columns
]

if EVALUACIONES_COMUNES:
    df_comparacion = comparar_ciclos(
        df_ciclo1=df_2025_2,
        df_ciclo2=df_2026_1,
        evaluaciones=EVALUACIONES_COMUNES,
        ciclo1_nombre="2025-2",
        ciclo2_nombre="2026-1",
    )
    display(df_comparacion)
else:
    print("⚠️ No hay evaluaciones comunes entre los dos ciclos para comparar.")
    df_comparacion = pd.DataFrame()

## 4. Gráfica de barras dobles por evaluación

In [ ]:
if not df_comparacion.empty:
    grafica_comparacion_ciclos(
        df_comparacion=df_comparacion,
        ciclo1_nombre="2025-2",
        ciclo2_nombre="2026-1",
        guardar_en=RAIZ / "data" / "2026-1" / "output" / "comparacion_ciclos_EA.png",
        figsize=(12, 6),
    )

## 5. Comparar distribución de NF entre ciclos

In [ ]:
if "NF" in df_2025_2.columns and "NF" in df_2026_1.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogramas superpuestos
    nf_2025_2 = df_2025_2["NF"].dropna()
    nf_2026_1 = df_2026_1["NF"].dropna()

    axes[0].hist(nf_2025_2, bins=20, alpha=0.6, label="2025-2", color="#4C72B0", edgecolor="white")
    axes[0].hist(nf_2026_1, bins=20, alpha=0.6, label="2026-1", color="#DD8452", edgecolor="white")
    axes[0].axvline(10.5, color="red", linestyle="--", label="Mín. aprobatorio")
    axes[0].set_title("Distribución de NF por ciclo")
    axes[0].set_xlabel("Nota Final")
    axes[0].set_ylabel("Número de alumnos")
    axes[0].legend()

    # Boxplot comparativo
    df_box = pd.DataFrame({
        "NF": pd.concat([nf_2025_2, nf_2026_1], ignore_index=True),
        "Ciclo": ["2025-2"] * len(nf_2025_2) + ["2026-1"] * len(nf_2026_1),
    })
    sns.boxplot(data=df_box, x="Ciclo", y="NF", ax=axes[1], palette=["#4C72B0", "#DD8452"])
    axes[1].axhline(10.5, color="red", linestyle="--", label="Mín. aprobatorio")
    axes[1].set_title("Boxplot de NF por ciclo")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(
        RAIZ / "data" / "2026-1" / "output" / "comparacion_NF.png",
        dpi=150, bbox_inches="tight"
    )
    plt.show()

    # Estadísticas comparativas de NF
    print("\nEstadísticas de NF:")
    print(f"  2025-2: media={nf_2025_2.mean():.2f}, mediana={nf_2025_2.median():.2f}, "
          f"aprobados={( nf_2025_2 >= 10.5).mean()*100:.1f}%")
    print(f"  2026-1: media={nf_2026_1.mean():.2f}, mediana={nf_2026_1.median():.2f}, "
          f"aprobados={(nf_2026_1 >= 10.5).mean()*100:.1f}%")
else:
    print("⚠️ La columna 'NF' no está disponible en uno o ambos ciclos.")

## 6. Análisis de alumnos que repiten el curso

In [ ]:
# Identificar alumnos que cursaron en ambos ciclos (por Código)
if "Código" in df_2025_2.columns and "Código" in df_2026_1.columns:
    codigos_2025_2 = set(df_2025_2["Código"].dropna().astype(str))
    codigos_2026_1 = set(df_2026_1["Código"].dropna().astype(str))

    repitentes = codigos_2025_2.intersection(codigos_2026_1)
    print(f"Alumnos en 2025-2: {len(codigos_2025_2)}")
    print(f"Alumnos en 2026-1: {len(codigos_2026_1)}")
    print(f"Alumnos que repiten (en ambos ciclos): {len(repitentes)}")

    if repitentes:
        # Comparar notas de repitentes entre ciclos
        df_rep_2025 = df_2025_2[df_2025_2["Código"].astype(str).isin(repitentes)].copy()
        df_rep_2026 = df_2026_1[df_2026_1["Código"].astype(str).isin(repitentes)].copy()

        cols_mostrar = ["Código"] + ["NF"] if "NF" in df_rep_2025.columns else ["Código"]

        df_repitentes_comparado = df_rep_2025[cols_mostrar].merge(
            df_rep_2026[["Código", "NF"]].rename(columns={"NF": "NF_2026_1"}),
            on="Código",
            how="inner",
        ).rename(columns={"NF": "NF_2025_2"})

        if "NF_2025_2" in df_repitentes_comparado.columns and "NF_2026_1" in df_repitentes_comparado.columns:
            df_repitentes_comparado["Mejora"] = (
                df_repitentes_comparado["NF_2026_1"] - df_repitentes_comparado["NF_2025_2"]
            ).round(2)
            df_repitentes_comparado = df_repitentes_comparado.sort_values("Mejora", ascending=False)

        print(f"\nComparación de notas para los {len(repitentes)} repitentes:")
        display(df_repitentes_comparado)
else:
    print("⚠️ La columna 'Código' no está disponible en uno o ambos ciclos.")